# Deep Embedded Clustering (DEC): algorithm walkthrough

Based on “Unsupervised Deep Embedding for Clustering Analysis” by Xie et al. (2016).

- Autoencoder pretraining initializes the latent representation.
- Student's t-distribution defines soft assignments.
- The auxiliary target strengthens confident assignments and adjusts for cluster frequency.
- KL divergence drives joint updates of the encoder and centroids.

**Objective: reproduce the DEC model and its learning procedure.**
Clustering scores are diagnostics, not acceptance thresholds. The implementation
uses the paper's `d–500–500–2000–10` encoder, greedy denoising pretraining,
pretrained mirrored decoder, Student-t assignments, frequency-adjusted targets,
and joint encoder/centroid optimization with momentum SGD. Targets are refreshed
in mini-batch updates, with the 0.1% assignment-change stopping criterion.

The local random-pixel and sklearn digits demonstrations retain short budgets.
Their data and preprocessing differ from the published benchmarks. The optional
[MNIST experiment](03_dec_mnist_paper_experiment.ipynb) uses the published
pretraining budget. Coordinate rankings and subset ablations are additional
diagnostics, not components of DEC.

**Execution pending.** Outputs from the earlier reconstruction-loss implementation
have been cleared. Run this notebook from top to bottom to generate results
with the corrected helper. See [implementation scope](../docs/dec_paper_alignment.md)
for the paper/code correspondence and differences from the authors' Caffe code.


### STEP 1 — Autoencoder Pretraining

Trains a stacked denoising autoencoder on input data.

This section implements a **greedy layer-wise pretraining** procedure for a deep autoencoder, followed by **end-to-end fine-tuning**.  
The final result is an encoder that maps input data into a compact latent space, ready for downstream tasks like clustering.

The pretraining procedure lives in `src/dec_pretraining.py`, shared by all three notebooks. It reuses `Encoder` and `FullAutoencoder` from `src/dec_model.py`, which also contains the DEC equations. The SGD clustering loop is shared in `src/dec_training.py`.

---

### 🔧 Configuration
- **Device (`DEVICE`)**: Uses GPU if available, otherwise CPU  
- **Batch size (`BATCH_SIZE`)**: Controls how many samples are processed at once  
- **Learning rate**: pretraining starts at 0.1 and decays by 10 every 20,000 updates; DEC uses 0.01
- **Updates**: `PRETRAIN_STEPS` per shallow layer and `FINETUNE_STEPS` for the full autoencoder  

---

### 1. Denoising Autoencoder (DAE)

A shallow autoencoder reconstructs its clean input after dropout corrupts both its input and hidden representation. It is the building block for greedy layer-wise pretraining.

- **Purpose**: Trains each layer to reconstruct its input from a noisy version  
- **Structure**:  
  - **Encoder**: Linear transformation to the next hidden dimension  
  - **Decoder**: Linear transformation to reconstruct the original input  
- **Activation**: ReLU after encoding and decoding, except the final encoder and first decoder, whose outputs are linear  
- **Training objective**: half the squared reconstruction error, averaged over samples and features (the authors' modified Caffe loss)  

This is the building block for greedy pretraining.

---

### 2. Stacked Encoder

Used to forward data through pretrained encoder layers.

- A container for a variable number of linear encoder layers  
- Built using `nn.ModuleList` so all layers are properly registered for:
  - Parameter optimization
  - Device transfers (`.to(DEVICE)`)
  - Saving/loading via `state_dict`  
- Applies activation inside the encoder `forward()` method, so the trained nonlinear mapping is preserved  

`Encoder` constructs a sequence of `nn.Linear` layers defined by the dimensions.
`nn.ModuleList` is used to register the layers as submodules, which ensures that their parameters
are included in `model.parameters()`, handled by `.to(device)`, and saved via `state_dict()`.
Unlike a plain Python list, `ModuleList` integrates with PyTorch’s internal module tracking system,
enabling proper parameter optimization and recursive operations.
Any time you want to manage a variable number of layers or modules, use `nn.ModuleList`
(or `nn.Sequential` if the sequential container itself fully defines the forward pass you need).

---

### 3. Greedy Layer-Wise Pretraining
1. For each target hidden dimension in `layer_dims`:
   - Train a shallow DAE on the current input  
   - Apply dropout to the input and hidden representation (denoising)  
   - Optimize with momentum SGD using half the squared error, averaged over samples and features  
2. After training:
   - Switch the shallow autoencoder to evaluation mode to disable dropout  
   - Transform the dataset with the trained encoder  
   - Use the transformed dataset as input for the next layer  
3. Retain the weights of both the encoder and decoder layers  

This initializes the full autoencoder. Reconstruction quality and clustering usefulness must be checked from the resulting diagnostics.

---

### 4. Full Autoencoder Assembly
- **Encoder**: `Encoder` module containing all pretrained encoder layers and activations  
- **Decoder**: Mirrors the encoder architecture and loads the trained shallow decoder weights in reverse layer order  
- **Forward pass**:
  - Encode through the learned nonlinear mapping  
  - Decode through the mirrored decoder, leaving the final reconstruction layer unclipped  

---

### 5. End-to-End Fine-Tuning
- Assemble the full autoencoder from the pretrained weights, without dropout  
- Train on the original dataset to minimize reconstruction error  
- Shuffle mini-batches each epoch; shuffling alone does not establish generalization  
- After training:
  - Switch encoder to inference mode (`.eval()`)  
  - Use encoder for extracting latent representations  

---

### 6. Diagnostic Preview
- Use `torch.no_grad()` for efficiency (no gradients tracked)  
- Compare final reconstruction MSE with a constant mean-vector baseline on the same training sample
- Preview latent statistics for the first ten samples; the following cell also prints standard deviations over the complete sample  

---

✅ **Outcome**: A pretrained encoder that maps high-dimensional input data into a compact latent space, initialized for clustering; its usefulness remains an empirical question.


In [ ]:
import os
import sys
from pathlib import Path

# Keep native numerical libraries conservative inside the notebook.
# This should run before importing torch, sklearn, numba, or umap.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMBA_NUM_THREADS", "1")

# The notebook may be opened from the repo root or from the notebooks folder.
# PROJECT_ROOT gives us one stable anchor for imports and saved artifacts.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
MODEL_PATH = PROJECT_ROOT / "outputs" / "dec_encoder_final.pt"
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib_cache"))

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import torch
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from dec_model import (
    DEVICE,
    Encoder,
    compute_soft_assignments,
    compute_target_distribution,
    encode_dataset,
    initialize_clusters,
    kl_loss,
)
from dec_pretraining import pretrain_autoencoder
from dec_training import train_dec
from dec_visualization import plot_final_tsne, plot_tsne
from latent_space_tools import compare_clustering_stages



In [ ]:
# Paper architecture; short budgets for tracing the algorithm locally.
# One step is one mini-batch optimizer update, not one full-data epoch.
LAYER_DIMS = [500, 500, 2000, 10]
N_CLUSTERS = 10
BATCH_SIZE = 256
PRETRAIN_STEPS = 400
FINETUNE_STEPS = 800
PRETRAIN_DROPOUT = 0.2

print(f"Using device: {DEVICE}")
print(f"Layer dimensions: {LAYER_DIMS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Pretraining updates per layer: {PRETRAIN_STEPS}")
print(f"Full-autoencoder updates: {FINETUNE_STEPS}")
print("Momentum SGD; short demonstration budget, not the paper benchmark.")


The variable `encoder` stores the trained encoder model returned by `pretrain_autoencoder()`.
This model is an `Encoder` object imported from `dec_model.py`: it contains the trained linear layers and the activation logic in its `forward()` method.

Later, when we write `encoder(X_tensor)`, we're calling the model on input data.
In PyTorch, models are callable: calling them triggers their `forward()` method as implemented in the `__call__()` method.

Even though we use the same name `encoder` inside and outside the function, there's no conflict:
the function's `encoder` is a local variable, and we just reuse the name when assigning the return value.
This is standard Python behavior and helps keep the code readable and intuitive.


In [ ]:
# Simulate data
np.random.seed(42)
torch.manual_seed(42)
X = np.random.rand(10000, 784).astype(np.float32)

# Pretrain encoder
# Return the encoder model, including its linear layers and activation behavior.
encoder = pretrain_autoencoder(
    X,
    layer_dims=LAYER_DIMS,
    batch_size=BATCH_SIZE,
    dropout_rate=PRETRAIN_DROPOUT,
    pretrain_steps=PRETRAIN_STEPS,
    finetune_steps=FINETUNE_STEPS,
    lr=0.1,
    momentum=0.9,
    lr_step_size=20000,
    lr_gamma=0.1,
    device=DEVICE,
)


### Diagnosis

# Encode the full dataset after pretraining.
# Keep a NumPy copy for plotting and a tensor for the shape diagnostic.
Z_np = encode_dataset(encoder, X, batch_size=BATCH_SIZE, device=DEVICE)
Z = torch.tensor(Z_np, dtype=torch.float32, device=DEVICE)
print("Latent shape:", Z.shape)
print("Full-sample latent standard deviations:", Z_np.std(axis=0))

# Expected output: torch.Size([10000, 10])


### Read the Pretraining Diagnostics

The `(10000, 10)` shape checks the encoder's interface. It does not establish
that the representation preserves useful information.

Compare the printed final reconstruction MSE against the mean-vector baseline.
Both are evaluated on the same training sample, with the same mean-squared-error
reduction. A positive baseline-minus-autoencoder difference indicates better
reconstruction than that constant predictor; it does not establish semantic
clusters or performance on unseen data.

Layer-wise losses reconstruct different targets at different scales, so a smaller
loss in a deeper layer is not evidence of progressively better representations.
Use the full-sample latent standard deviations to inspect spread. The preview
printed by the helper covers only the first ten observations.

These inputs are independent random pixels with no designed semantic groups.
Record the observed baseline comparison and latent spread after execution before
drawing conclusions about this run.


#### STEP 1.1 — UMAP / PCA Visual Check

UMAP (Uniform Manifold Approximation and Projection) is a dimensionality reduction technique used to visualize and analyze high-dimensional data in a lower-dimensional space.

In this local environment, the UMAP stack can hard-crash the notebook kernel because it goes through native `numba` / OpenMP code. The warning
`n_jobs value 1 overridden to 1 by setting random_state` is harmless, and the `omp_set_nested` line is also only an OpenMP deprecation notice. The kernel restart is the real problem.

For that reason, the cell below uses PCA by default for the quick diagnostic plot. PCA is less manifold-aware than UMAP, but it is stable and still useful for checking whether the latent space has obvious spread or collapse. If the environment is stable after package fixes, set `VISUAL_METHOD = "umap"` in the next cell.

Technical Assumptions Behind UMAP:

1.	Data Lies on a Riemannian Manifold
	•	UMAP assumes the high-dimensional data points are sampled from a manifold — a smooth, curved geometric space embedded within the high-dimensional ambient space.
	•	This means local neighborhoods in the data are meaningful and reflect intrinsic geometric structure.

2.	Local Connectivity / Locality Structure Is Meaningful
	•	The relationships between a point and its nearest neighbors carry key information about the data structure.
	•	UMAP models these local relationships via a weighted graph (fuzzy simplicial complex).

3.	Metric Space Assumption
	•	The input data space is a metric space with a defined distance function (e.g., Euclidean distance).
	•	Distances meaningfully reflect similarity or dissimilarity.



A projection can show groups or curves even when the input has no semantic clusters. Its appearance depends on the projection and its parameters. Inspect latent variance and the quantitative diagnostics alongside the plot; visible separation alone does not validate the learned representation.


In [ ]:
# Convert latent tensor Z from GPU to CPU and then to NumPy.
if isinstance(Z, torch.Tensor):
    Z_np = Z.detach().cpu().numpy()
else:
    Z_np = np.asarray(Z)

# Native UMAP crashes can restart the kernel before Python can catch an error.
# Keep PCA as the default stable visual diagnostic. Change to "umap" only after
# the local OpenMP/numba/UMAP environment is known to be stable.
VISUAL_METHOD = "pca"  # options: "pca", "umap"
MAX_VISUAL_POINTS = 3000

Z_visual_source = Z_np.astype(np.float32, copy=False)
if len(Z_visual_source) > MAX_VISUAL_POINTS:
    rng = np.random.default_rng(42)
    visual_idx = np.sort(rng.choice(len(Z_visual_source), size=MAX_VISUAL_POINTS, replace=False))
    Z_visual_source = Z_visual_source[visual_idx]
else:
    visual_idx = np.arange(len(Z_visual_source))

if VISUAL_METHOD == "umap":
    import umap.umap_ as umap

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        metric="euclidean",
        low_memory=True,
        n_jobs=1,
        random_state=42,
    )
    Z_visual = reducer.fit_transform(Z_visual_source)
    visual_title = "UMAP Projection of Latent Space"
else:
    from sklearn.decomposition import PCA

    reducer = PCA(n_components=2, random_state=42)
    Z_visual = reducer.fit_transform(Z_visual_source)
    visual_title = "PCA Projection of Latent Space"

# Plot the 2D projection.
plt.figure(figsize=(8, 6))
sns.scatterplot(x=Z_visual[:, 0], y=Z_visual[:, 1], s=10, alpha=0.6)
plt.title(visual_title)
plt.grid(True)
plt.show()


### STEP 2 — Cluster Initialization

Transform the raw input data X into a latent representation Z using the pretrained encoder from Step 1, and then apply KMeans to initialize the cluster centroids {μ₁, ..., μ_k} using k-means for Deep Embedded Clustering (DEC).

This seeds DEC with initial centroids in the embedded space. Whether those centroids correspond to useful groups depends on the learned representation and data.


In [ ]:
# initialize_clusters(...) is imported from src/dec_model.py.
# This cell is kept here as the Step 2 implementation pointer so the notebook
# structure still follows the DEC paper: first encode X into Z, then run KMeans
# in that latent space to initialize the centroids mu.


In [ ]:
# Step 2: Initialize clusters
Z, mu, kmeans_labels = initialize_clusters(
    encoder,
    X,
    k=N_CLUSTERS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

# Sanity check
print("Z sample:", Z[:2])
print("mu sample:", mu[:2])


### Step 3 – Soft Assignment via Student’s t-Distribution


This step transforms each encoded point zᵢ and each cluster centroid μⱼ into a soft assignment probability qᵢⱼ, using the Student’s t-distribution as a kernel.

In DEC, qᵢⱼ is interpreted as the probability that sample i belongs to cluster j. Unlike hard KMeans assignments, these probabilities allow gradient-based learning in subsequent steps.

---

#### Step-by-Step Process

**Step 1 – Distance Matrix**  
Compute the full squared Euclidean distance matrix D of shape `(n_samples, n_clusters)`.  
Each entry $ D[i, j] $ is the squared distance between a latent vector zᵢ and centroid μⱼ.  
We use **NumPy broadcasting** instead of explicit nested loops for efficiency:


squared_dist = np.sum((Z[:, np.newaxis, :] - mu[np.newaxis, :, :])**2, axis=2)


**Step 2 – Convert Distances to Similarities**  
Apply the Student’s t-distribution kernel to convert distances into **unnormalized affinities**:

$$ 
q_{ij} \propto \left(1 + \frac{\|z_i - \mu_j\|^2}{\alpha}\right)^{-\frac{\alpha+1}{2}}
$$

- **Parameter $alpha$** controls the shape of the kernel:
  - $alpha = 1$: Cauchy-like kernel, heavy-tailed, tolerant of outliers  
  - Larger $alpha$: decays faster, closer to Gaussian  

**Step 3 – Normalize to Probabilities**  
Normalize row-wise so that each sample’s probabilities sum to 1
$Q[i, j] ≈$ probability of sample i belonging to cluster j

Q = unnormalized_affinity / np.sum(unnormalized_affinity, axis=1, keepdims=True)

--- 

#### Shape Evolution in Soft Assignment Computation

Think of the data flow as reshaping physical objects to make them fit together for interaction:

- **Latent space**
  - `Z`: (n_samples, latent_dim)  
    Each sample is a **point in latent space**, like particles in a 10‑dimensional box.  
  - `mu`: (n_clusters, latent_dim)  
    Each centroid is a **reference point** (like anchors) in the same box.

- **Broadcasting → Expanding dimensions**
  ```python
  squared_dist = np.sum((Z[:, np.newaxis, :] - mu[np.newaxis, :, :])**2, axis=2)
  ```

The broadcast difference has shape `(n_samples, n_clusters, latent_dim)`.
Summing over the last axis gives one squared distance per sample and centroid.
The paper fixes `alpha = 1`, as this notebook does.


In [ ]:
# compute_soft_assignments(...) is imported from src/dec_model.py.
# It implements the Student's t-distribution kernel from the DEC paper and
# returns Q, where each row is a probability distribution over clusters.


In [ ]:
# Step 3: Soft Assignment
Q = compute_soft_assignments(Z, mu)

Expected behavior

Q.shape: (n_samples, k) — matches data and cluster count

Each row sums to 1 — valid probability distribution

Q values are soft, not hard — e.g., [0.1, 0.7, 0.05, ...]

Higher value means stronger assignment to that cluster

Q[:3]: Each row is a probability distribution over clusters

If all values are close to 0.1, the model is uncertain. If one value dominates, the model is confident.


### STEP 3.2 — Compute Target Distribution


In this step, we take the soft assignment matrix Q (from Step 3) and compute a sharpened target distribution P. This distribution places more emphasis on confident assignments and helps reinforce cluster purity during training.

Instead of treating Q as the optimization target, the DEC algorithm computes a modified P to pull embeddings toward high-confidence cluster centers.


In [ ]:
# compute_target_distribution(...) is imported from src/dec_model.py.
# It squares Q to sharpen confident assignments, divides by soft cluster
# frequency to avoid letting large clusters dominate, and normalizes each row.


In [ ]:
P = compute_target_distribution(Q)

### Check the Target Distribution

`P` should have the same shape as `Q`, non-negative entries, and rows that sum
to one. Inspect the printed minimum, maximum, and sample rows from this run.

Squaring affinities strengthens confident assignments, while division by the
soft cluster frequencies balances their contributions. Together these operations
do not guarantee that every row becomes more peaked, that every maximum grows,
or that the preferred cluster stays unchanged. A minimum need not increase.
The entropy comparison below measures the actual result for this sample.

Near-uniform `Q` can yield near-uniform `P`; this is particularly relevant to the
random-pixel example. Interpret the observed values after execution.


#### STEP 3.2.1 — t-SNE Check

t-SNE projects the latent vectors into two dimensions, emphasizing local
neighborhoods. Each point represents one sample; color shows its cluster ID.

This is a visualization, not a clustering-quality test. Coloring by clusters
computed from these same vectors can produce separated colors without establishing
semantic structure. Read this plot alongside confidence, latent spread, and,
for the later digits example, external evaluation metrics.


In [ ]:
# plot_tsne(...) is imported from src/dec_visualization.py.
# Keeping the plotting call in the notebook makes the diagnostic flow explicit,
# while the reusable plotting helper stays outside the DEC modeling module.


In [ ]:
plot_tsne(Z, labels=kmeans_labels, title="t-SNE: Colored by KMeans Clusters")

### Compare KMeans Labels with the Largest Soft Assignment

For fixed `Z` and `mu`, the Student-t affinity decreases monotonically with
squared Euclidean distance. Normalizing a row does not change its ordering.
Therefore `argmax(Q)` selects the nearest centroid, just as the KMeans prediction
rule does, except for ties and numerical precision effects.

The next cell prints agreement counts and shows the same embedding colored by
`argmax(Q)`. Differences at this stage do not demonstrate non-convex structure.
DEC can change the clustering later by updating the encoder and the centroids.


In [ ]:
Q_labels = np.argmax(Q, axis=1)
# Float64 distances help inspect ties in nearly collapsed float32 embeddings.
squared_distances = np.sum(
    (Z.astype(np.float64)[:, None, :] - mu.astype(np.float64)[None, :, :]) ** 2,
    axis=2,
)
nearest_centroid_labels = squared_distances.argmin(axis=1)
print(
    "Q versus nearest-centroid disagreements:",
    np.count_nonzero(Q_labels != nearest_centroid_labels),
)
print(
    "Q versus KMeans disagreements:",
    np.count_nonzero(Q_labels != kmeans_labels),
)
plot_tsne(Z, labels=Q_labels, title="t-SNE: Colored by Soft Assignments (Q)")

### STEP 3.3 — KL Divergence Loss

Goal: Compute KL Divergence Between P and Q

The DEC algorithm updates the encoder by minimizing the KL divergence between the auxiliary target distribution P (sharpened assignments) and the current soft assignments Q.

This loss encourages the encoder to map samples into regions where confident cluster assignments (from P) are matched by the model’s actual assignments (Q).


In [ ]:
# kl_loss(...) is imported from src/dec_model.py.
# It computes KL(P || Q) with small clamps for numerical stability.


In [ ]:
# Convert from numpy to torch
P_torch = torch.tensor(P, dtype=torch.float32).to(DEVICE)
Q_torch = torch.tensor(Q, dtype=torch.float32).to(DEVICE)

# Compute KL divergence loss
loss = kl_loss(P_torch, Q_torch)
print(f"🔥 KL divergence loss: {loss.item():.6f}")

### Interpret KL Loss Together with the Assignments

There is no universal KL-loss threshold for good clustering. This implementation
reports `KL(P || Q)` averaged over samples; the paper writes the summed objective.
A low value means that `Q` matches the current target `P`, which can also happen
for uninformative assignments.

Read the loss alongside assignment confidence, entropy, cluster sizes, and the
later digits evaluation. During training, `P` is periodically refreshed, so loss
can jump even when the optimizer is behaving correctly. Losses on either side of
a target refresh refer to different objectives.


In [ ]:
# Check Distribution of Q
# If all are close to 0.1 → Soft assignments are too flat.
print("Q max per sample (top 5):", torch.max(Q_torch, dim=1).values[:5].cpu().numpy())

In [ ]:
# Compare Entropy of P and Q
# Lower entropy means more confident predictions (sharper distributions).
# Frequency normalization means P need not have lower entropy in every row.

entropy_P = -torch.sum(torch.special.xlogy(P_torch, P_torch), dim=1).mean()
entropy_Q = -torch.sum(torch.special.xlogy(Q_torch, Q_torch), dim=1).mean()
print(f"P entropy: {entropy_P.item():.4f} | Q entropy: {entropy_Q.item():.4f}")

### Step 3.4 — Training Step
Fine-tune the encoder so that it learns to shape the latent space Z in a way that improves cluster assignment
by minimizing the KL divergence loss.

This is done iteratively:

1.	Forward pass to compute Z → Q

2.	Compute target distribution P from Q

3.	Compute KL(P‖Q) loss

4.	Backpropagate loss through the encoder and the cluster centroids

5.	Update parameters

6.	Refresh the full-sample P every `update_interval` mini-batch updates; stop when fewer than 0.1% of hard assignments change between refreshes, or at `max_steps`


In [ ]:
# The loop in src/dec_training.py updates both the encoder
# and centroids with momentum SGD. P is detached and refreshed over all samples
# every update_interval mini-batch steps, using the existing DEC equations.


In [ ]:
Z, mu, kmeans_labels = initialize_clusters(
    encoder,
    X,
    k=N_CLUSTERS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

dec_result = train_dec(
    encoder,
    X,
    mu,
    batch_size=BATCH_SIZE,
    max_steps=1000,
    update_interval=40,
    lr=0.01,
    momentum=0.9,
    alpha=1.0,
    tol=0.001,
    device=DEVICE,
)

# train_dec mutates encoder in place and returns the updated centroids/labels
# so later diagnostics use the DEC-trained state, not the old KMeans centroids.
encoder = dec_result.encoder
mu = dec_result.centroids
dec_labels = dec_result.labels
loss_history = dec_result.loss_history


### Final t-SNE Visualization of DEC Embeddings

Inspect the reported stopping reason separately from clustering quality. A small
assignment change can occur in a degenerate state. The projection is a visual
diagnostic; verify cluster occupancy and assignment probabilities alongside it.
Its appearance is not an acceptance criterion for structural replication.


In [ ]:
Z_final = plot_final_tsne(encoder, X)

In [ ]:
# Optional Diagnostics After Training to assess if Q is sharp or too uniform

Z_after_dec = encode_dataset(encoder, X, batch_size=BATCH_SIZE, device=DEVICE)
Q_np = compute_soft_assignments(Z_after_dec, mu)
max_probs = Q_np.max(axis=1)

print("Q confidence stats:")
print("Min:", np.min(max_probs))
print("Max:", np.max(max_probs))
print("Mean:", np.mean(max_probs))


## PART 1B — A Full DEC Pipeline on a Classic Dataset: Digits

The random dataset above is useful because it lets us inspect every mathematical step without worrying about labels or domain details. But random pixels do not contain real cluster structure, so the result is not a meaningful test of whether DEC can discover semantic groups.

Here we repeat the same DEC workflow on the classic `sklearn` digits dataset. It is a small handwritten-digit dataset with 8×8 grayscale images. It is not the full MNIST dataset, but it has the same spirit: each row is an image, and each image belongs to one of ten digit classes.

Important point: the labels are used only for evaluation after clustering. DEC itself still trains as an unsupervised method.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

# Load a classic handwritten-digits dataset.
digits = load_digits()
X_digits_raw = digits.data.astype(np.float32)
y_digits = digits.target.astype(int)
digits_images = digits.images

# A compact preprocessing pipeline:
# 1. MinMaxScaler keeps values in [0, 1], similar to normalized image pixels.
# 2. VarianceThreshold removes pixels that never change across the dataset.
digits_preprocess = Pipeline(
    steps=[
        ("scale_to_unit_interval", MinMaxScaler()),
        ("drop_constant_pixels", VarianceThreshold(threshold=0.0)),
    ]
)
X_digits = digits_preprocess.fit_transform(X_digits_raw).astype(np.float32)

print("Raw digits shape:", X_digits_raw.shape)
print("Preprocessed digits shape:", X_digits.shape)
print("Number of true digit classes:", len(np.unique(y_digits)))
print("Pixel value range after preprocessing:", (X_digits.min(), X_digits.max()))


### Digits Step 1 — Look at the Data Before Clustering

Before training any model, it helps to look at the input. Each image is only 8×8 pixels, so the data is much smaller than modern MNIST images. It provides a small local example for tracing the implementation; it is not the paper's MNIST benchmark.

The model will not see these labels during training. The labels are shown here so we can understand the dataset and later evaluate whether the unsupervised clusters align with actual digit identities.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit_value, ax in enumerate(axes.ravel()):
    first_idx = np.where(y_digits == digit_value)[0][0]
    ax.imshow(digits_images[first_idx], cmap="gray_r")
    ax.set_title(f"Digit {digit_value}")
    ax.axis("off")

plt.suptitle("One Example Image per Digit Class", y=1.02)
plt.tight_layout()
plt.show()


### Digits Step 2 — Define the DEC Pipeline

For this dataset, the full workflow is:

1. Load the images as tabular vectors.
2. Scale the pixel values to a stable numeric range.
3. Pretrain an autoencoder to compress each image into a lower-dimensional representation.
4. Run KMeans in that latent space to initialize cluster centers.
5. Fine-tune the encoder and centroids with DEC's KL-divergence objective.
6. Evaluate the final clusters against labels that were not used during training.

The encoder uses the paper's `d–500–500–2000–10` architecture. Only `d` changes with the input; the short budgets and sklearn digits preprocessing remain local demonstration choices.


In [ ]:
DIGITS_LAYER_DIMS = [500, 500, 2000, 10]
DIGITS_N_CLUSTERS = 10
DIGITS_BATCH_SIZE = 256
DIGITS_PRETRAIN_STEPS = 200
DIGITS_FINETUNE_STEPS = 400
DIGITS_DEC_STEPS = 1000

print("Digits DEC configuration")
print("Input dimension:", X_digits.shape[1])
print("Layer dimensions:", DIGITS_LAYER_DIMS)
print("Clusters:", DIGITS_N_CLUSTERS)
print("Batch size:", DIGITS_BATCH_SIZE)


In [ ]:
# Seed this example independently of the earlier random-data walkthrough.
np.random.seed(42)
torch.manual_seed(42)

# Step 1: autoencoder pretraining on real handwritten-digit data.
digits_encoder = pretrain_autoencoder(
    X_digits,
    layer_dims=DIGITS_LAYER_DIMS,
    batch_size=DIGITS_BATCH_SIZE,
    dropout_rate=PRETRAIN_DROPOUT,
    pretrain_steps=DIGITS_PRETRAIN_STEPS,
    finetune_steps=DIGITS_FINETUNE_STEPS,
    lr=0.1,
    momentum=0.9,
    lr_step_size=20000,
    lr_gamma=0.1,
    device=DEVICE,
)

# Step 2: initialize DEC clusters with KMeans in the pretrained latent space.
digits_Z_init, digits_mu_init, digits_kmeans_labels = initialize_clusters(
    digits_encoder,
    X_digits,
    k=DIGITS_N_CLUSTERS,
    batch_size=DIGITS_BATCH_SIZE,
    device=DEVICE,
)

# Step 3: run the DEC refinement phase.
digits_dec_result = train_dec(
    digits_encoder,
    X_digits,
    digits_mu_init,
    batch_size=DIGITS_BATCH_SIZE,
    max_steps=DIGITS_DEC_STEPS,
    update_interval=40,
    lr=0.01,
    momentum=0.9,
    alpha=1.0,
    tol=0.001,
    device=DEVICE,
)

digits_encoder = digits_dec_result.encoder
digits_mu = digits_dec_result.centroids
digits_dec_labels = digits_dec_result.labels
digits_loss_history = digits_dec_result.loss_history
digits_Z_final = encode_dataset(digits_encoder, X_digits, batch_size=DIGITS_BATCH_SIZE, device=DEVICE)

print("Final digits latent shape:", digits_Z_final.shape)


### Digits Step 3 — Evaluate the Clusters

Clustering labels are arbitrary. For example, DEC cluster `0` does not automatically mean digit `0`. A cluster might correspond to digit `7`, digit `3`, or a mixture of several digits.

That is why we use three complementary metrics:

- **Adjusted Rand Index (ARI)**: compares pairs of samples and corrects for chance.
- **Normalized Mutual Information (NMI)**: measures shared information between clusters and true labels.
- **Hungarian matched accuracy**: finds the best one-to-one mapping between cluster IDs and digit labels, then reports accuracy.

The first two are standard clustering metrics. The matched accuracy is intuitive for a ten-class digit example, but it should be interpreted carefully because it uses the labels after training.


In [ ]:
stage_to_labels = {
    "KMeans on pretrained Z": digits_kmeans_labels,
    "DEC refined clusters": digits_dec_labels,
}
digits_metrics, digits_clustering_details = compare_clustering_stages(y_digits, stage_to_labels)

digits_kmeans_details = digits_clustering_details["KMeans on pretrained Z"]
digits_dec_details = digits_clustering_details["DEC refined clusters"]

digits_kmeans_acc = digits_kmeans_details.matched_accuracy
digits_kmeans_map = digits_kmeans_details.cluster_to_label
digits_kmeans_aligned = digits_kmeans_details.aligned_labels
digits_kmeans_cm = digits_kmeans_details.confusion_matrix

digits_dec_acc = digits_dec_details.matched_accuracy
digits_cluster_to_label = digits_dec_details.cluster_to_label
digits_aligned_labels = digits_dec_details.aligned_labels
digits_dec_cm = digits_dec_details.confusion_matrix

display(digits_metrics.style.format({"ARI": "{:.3f}", "NMI": "{:.3f}", "matched_accuracy": "{:.3f}"}))
print("Cluster -> digit mapping learned only for evaluation:", digits_cluster_to_label)


In [ ]:
# Visualize the before/after embeddings with PCA for a stable, notebook-friendly plot.
digits_projection_model = PCA(n_components=2, random_state=42)
digits_Z_init_2d = digits_projection_model.fit_transform(digits_Z_init)
digits_Z_final_2d = PCA(n_components=2, random_state=42).fit_transform(digits_Z_final)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

scatter0 = axes[0].scatter(
    digits_Z_init_2d[:, 0],
    digits_Z_init_2d[:, 1],
    c=y_digits,
    cmap="tab10",
    s=12,
    alpha=0.75,
)
axes[0].set_title("Pretrained Latent Space\nColored by True Digit")
axes[0].set_xlabel("PCA 1")
axes[0].set_ylabel("PCA 2")

scatter1 = axes[1].scatter(
    digits_Z_final_2d[:, 0],
    digits_Z_final_2d[:, 1],
    c=digits_dec_labels,
    cmap="tab10",
    s=12,
    alpha=0.75,
)
axes[1].set_title("DEC Latent Space\nColored by DEC Cluster")
axes[1].set_xlabel("PCA 1")
axes[1].set_ylabel("PCA 2")

sns.heatmap(digits_dec_cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[2])
axes[2].set_title("True Digit vs DEC Cluster")
axes[2].set_xlabel("DEC cluster")
axes[2].set_ylabel("True digit")

plt.tight_layout()
plt.show()


### Digits Step 4 — Interpret the Execution

Check that the encoder and centroids were jointly updated, targets were refreshed
using all samples, and the log distinguishes convergence from budget exhaustion.
These are checks of the DEC procedure. The metric table describes this short
local experiment; low accuracy alone does not imply an implementation error.

The before/after PCA panels use separate fitted projections. Neither the apparent
spacing nor colors alone establishes recovery of digit classes. Paper benchmark
comparisons belong in the optional MNIST experiment.


### Save and Reload the Random-Data Encoder

This persistence check saves the encoder from the random-data walkthrough and
reloads it as `reloaded_encoder`. It checks encoder persistence only; it does not
save the digit model or the DEC centroids required for complete cluster inference.


In [ ]:
# Save encoder for reuse
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(encoder.state_dict(), MODEL_PATH)
print(f"Encoder model saved to {MODEL_PATH}")

In [ ]:
# Code to reload the encoder without overwriting the active notebook encoder.
# This is useful when you want to compare a saved model with the current model.
reloaded_encoder = Encoder(input_dim=X.shape[1], layer_dims=LAYER_DIMS).to(DEVICE)
reloaded_encoder.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
reloaded_encoder.eval()


## Next Notebook — Latent Space Diagnostics

The latent-space analysis has been split into a separate notebook:

`02_dec_latent_space_diagnostics.ipynb`

That notebook focuses specifically on the learned DEC embedding `Z`: how to score latent dimensions, profile clusters, and test whether the coordinates with the highest individual separation scores preserve clustering structure.
